# FFM_3_3 — mesh2z：端到端训练的 PointNet（探索性）

> **探索性实验。** FFM_3_2 的停止规则已经触发（CV-R²(z) 0.076 < 0.10）。本 notebook 回答一个
> 单独的问题——**训练出来的点云特征，会不会比随机特征（RFF 均值）好？**——结果不改写 FFM_3_2
> 的结论，写进 log 时标注为探索性。

**和 FFM_3_2 完全可比**
- 同样的 164 个开发集受试者、**同样的外层 10 折**（从 FFM_3_2 导出的 `oof_ridge.npz` 读入）
- 同样的目标（每只耳的 z）、同样的四个端到端读数，外加 LSD
- ridge 的结果在这里用**同一套代码**重新打分，逐人配对比较，并给 bootstrap 置信区间

**防止「在噪声里挑最大值」**
- epoch 数、结构、学习率**跑之前固定**（Cell 1），不按验证分挑 epoch
- 每一折同时训练一个**打乱目标的对照**（训练集内「人 ↔ z」随机配对），在真实目标上评估
- 每 10 个 epoch 记录训练/验证 R²，**只用来画图**

**模型**：每只耳的耳廓点云（相对耳道，右耳已镜像）→ 逐点共享 MLP → max + mean 池化 → MLP 头 → z 的折内
主成分（K 个）→ 还原成 z。两种输入编码：原始 xyz，或 RFF（σ 单位 mm⁻¹）。

**输出**：`results.json`、学习曲线图，写到 `OUT_DIR`。

In [ ]:
# ==========================================================================
# Cell 0 — 环境 · 路径
# ==========================================================================
import os, sys, json, csv, zipfile, hashlib, time, math, warnings
warnings.filterwarnings('ignore')
import numpy as np

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive'
    SAVE_DIR = f'{DRIVE}/HRTF_Checkpoints'
    os.system('pip install healpy -q')
    PATHS = dict(
        latents = f'{SAVE_DIR}/mesh2z_sonicom_latents_v1.npz',
        oof     = f'{SAVE_DIR}/mesh2z_rff_v1/oof_ridge.npz',
        samples = f'{DRIVE}/ELEC5020/SONICOM/mesh2z_samples',
        ckpt    = f'{SAVE_DIR}/FFM26_best.pth',
        sonicom = f'{DRIVE}/ELEC5020/SONICOM.zip',
    )
    OUT_DIR = f'{SAVE_DIR}/mesh2z_pointnet_v1'
else:
    H = os.path.expanduser('~/Documents')
    R = f'{H}/26S2/ELEC5021/hrtf-GitHub'
    PATHS = dict(
        latents = f'{R}/mesh2z/output/mesh2z_sonicom_latents_v1.npz',
        oof     = os.environ.get('MESH2Z_OOF', f'{R}/mesh2z/output/rff_v1/oof_ridge.npz'),
        samples = f'{H}/26S1/ELEC5020/SONICOM/mesh2z_samples',
        ckpt    = os.environ.get('FFM26_CKPT', f'{R}/results/FFM26_best.pth'),
        sonicom = f'{H}/26S1/ELEC5020/SONICOM',
    )
    OUT_DIR = os.environ.get('MESH2Z_OUT', f'{R}/mesh2z/output/pointnet_v1')
os.makedirs(OUT_DIR, exist_ok=True)
for k, p in PATHS.items():
    print(f'{k:8s} {"OK " if os.path.exists(p) else "★缺失"} {p}')
assert all(os.path.exists(p) for p in PATHS.values()), \
    '先放好缺失的文件（oof_ridge.npz 由新版 FFM_3_2 的 Cell 12 生成）'

In [ ]:
# ==========================================================================
# Cell 1 — 配置（★ 跑之前定死，跑完不改）
# ==========================================================================
CFG = dict(
    VARIANTS    = ['xyz', 'rff'],          # 逐点输入：原始坐标 / RFF 编码
    XYZ_SCALE   = 50.0,                    # mm -> 约 ±1
    RFF_SIGMAS  = [0.01, 0.02, 0.05],      # mm⁻¹；FFM_3_2 里最好的单尺度在 0.02–0.05
    RFF_M       = 64,                      # 每个 σ 的频率数 -> 3 × 128 = 384 维
    N_PTS       = 2048,                    # 每次前向从 2 万个耳廓点里随机抽
    N_EVAL_DRAWS= 4,                       # 评估时用 4 组固定点子集取平均
    JITTER_MM   = 1.0,                     # 训练时整只耳随机平移，模拟耳道定位误差
    WIDTHS      = (64, 128, 256),          # 逐点共享 MLP
    HEAD        = 256,
    DROPOUT     = 0.3,
    K_PC        = 64,                      # 预测 z 的前 K 个折内主成分
    EPOCHS      = 200,                     # ★ 固定；不按验证分挑 epoch
    BATCH       = 32,
    LR          = 1e-3,
    WD          = 1e-2,
    EVAL_EVERY  = 10,
    SHUFFLE_CONTROL = True,
    N_BOOT      = 2000,
    SEED        = 0,
    NOTCH_HZ    = (6000, 12000),
    DID         = 2,
)
print(json.dumps(CFG, indent=1))

In [ ]:
# ==========================================================================
# Cell 2 - 模型定义（与 FFM_2_3 / FFM_2_6 逐字相同，构建时复制进来）
# ==========================================================================
import numpy as np
import torch
import torch.nn as nn
import healpy as hp


# ---------------------------------------------- spherical convolution
class OSLOConv(nn.Module):
    """Anisotropic convolution on HEALPix: each pixel plus its 8 neighbours, 9 independent weights."""
    def __init__(self, in_channels, out_channels, nside=16):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        neighbors = hp.get_all_neighbours(nside, np.arange(self.npix))
        rf = np.concatenate([neighbors, np.arange(self.npix)[np.newaxis, :]], axis=0)
        rf[rf == -1] = self.npix
        self.register_buffer('neighborhood', torch.tensor(rf, dtype=torch.int64))
        self.weight = nn.Parameter(torch.empty(out_channels, in_channels, 9))
        self.bias = nn.Parameter(torch.empty(out_channels))
        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))
        nn.init.zeros_(self.bias)

    def forward(self, x):
        B, C, _ = x.shape
        pad = torch.zeros(B, C, 1, device=x.device, dtype=x.dtype)
        g = torch.cat([x, pad], dim=2)[:, :, self.neighborhood]
        return torch.einsum('bcqn,ocq->bon', g, self.weight) + self.bias.view(1, -1, 1)


class OSLOResBlock(nn.Module):
    """Frequency-isolated residual block, operating on (B*F, C, N).
    * v2: ReLU -> GELU. HRTF log-magnitude is bipolar (negative means attenuation), so
      ReLU truncates the negative half; GELU is smooth, keeps negatives, and matches the fc layers."""
    def __init__(self, in_channels, out_channels, nside=16, num_groups=8):
        super().__init__()
        self.shortcut = (nn.Conv1d(in_channels, out_channels, 1)
                         if in_channels != out_channels else nn.Identity())
        self.conv1 = OSLOConv(in_channels, out_channels, nside)
        self.gn1 = nn.GroupNorm(num_groups, out_channels)
        self.conv2 = OSLOConv(out_channels, out_channels, nside)
        self.gn2 = nn.GroupNorm(num_groups, out_channels)
        self.act = nn.GELU()

    def forward(self, x):
        res = self.shortcut(x)
        out = self.act(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        return self.act(out + res)


# ---------------------------------------------- per-dataset FiLM
class DatasetFiLM(nn.Module):
    """Learn one PER-FREQUENCY scale/shift curve for each dataset.

    ** Constraint 1: frequency only, NEVER direction.
      A per-frequency affine map is one EQ curve (93 dof per dataset). All it can absorb is
      "colouration common to the whole dataset and independent of direction" — loudspeaker and
      microphone response, equalisation convention, overall level. By construction it CANNOT
      express a direction-dependent difference, so population anatomy (ear-size distributions
      differ across databases) must still live in z. Contrast: subtracting a per-database mean
      over (frequency x direction) is 3072x93 dof, enough to remove the real population structure
      as well — exactly the wrong direction Craig pointed at. A factor of 3072 in expressive power.

    ** Constraint 2: centre across datasets.
      Without centring, log_gamma/beta learn a global EQ that cancels against the network's own
      output gain — a redundant degree of freedom. After the cross-database mean is removed FiLM
      expresses only RELATIVE differences; the absolute part stays in the network. Exactly the identity when num_datasets=1.
    """
    def __init__(self, num_datasets, num_bins):
        super().__init__()
        self.num_datasets = num_datasets
        self.log_gamma = nn.Parameter(torch.zeros(num_datasets, num_bins))
        self.beta = nn.Parameter(torch.zeros(num_datasets, num_bins))

    def params(self, dataset_id):
        lg = self.log_gamma - self.log_gamma.mean(0, keepdim=True)
        bt = self.beta - self.beta.mean(0, keepdim=True)
        return torch.exp(lg[dataset_id]).unsqueeze(-1), bt[dataset_id].unsqueeze(-1)

    def forward(self, x, dataset_id):
        g, b = self.params(dataset_id)
        return x * g + b

    def inverse(self, x, dataset_id):
        g, b = self.params(dataset_id)
        return (x - b) / g


# ---------------------------------------------- encoder / decoder

class Isolated_OSLO_Encoder(nn.Module):
    """Encode each frequency bin independently -> latent (B, F, latent_dim)"""
    def __init__(self, nside=16, latent_dim_per_freq=64):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        self.latent_dim = latent_dim_per_freq
        self.enc1 = OSLOResBlock(1, 16, nside)
        self.enc2 = OSLOResBlock(16, 32, nside)
        self.fc_encode = nn.Sequential(
            nn.Flatten(), nn.Linear(32 * self.npix, 512), nn.GELU(),
            nn.Linear(512, self.latent_dim))

    def forward(self, x):
        B, F, N = x.shape
        z = self.fc_encode(self.enc2(self.enc1(x.reshape(B * F, 1, N))))
        return z.reshape(B, F, self.latent_dim)


class Conditioned_OSLO_Decoder(nn.Module):
    """FFM frequency-conditioned decoder + per-dataset FiLM on the output"""
    def __init__(self, nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                 num_datasets=1, num_bins=93):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        self.latent_dim = latent_dim_per_freq
        self.freq_embed_dim = freq_embed_dim
        self.ffm_B = nn.Parameter(torch.randn(freq_embed_dim // 2))
        self.fc_decode = nn.Sequential(
            nn.Linear(self.latent_dim + freq_embed_dim, 512), nn.GELU(),
            nn.Linear(512, 32 * self.npix))
        self.dec1 = OSLOResBlock(32, 16, nside)
        self.dec_final = OSLOConv(16, 1, nside)
        self.film = DatasetFiLM(num_datasets, num_bins)

    def get_freq_embedding(self, num_freqs, device):
        f = torch.arange(num_freqs, device=device).float() / (num_freqs - 1)
        proj = 2 * np.pi * f.unsqueeze(1) * self.ffm_B.unsqueeze(0)
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=1)

    def forward(self, z, dataset_id=None):
        B, F, _ = z.shape
        fe = self.get_freq_embedding(F, z.device).unsqueeze(0).expand(B, -1, -1)
        h = torch.cat([z, fe], dim=-1).reshape(B * F, -1)
        h = self.fc_decode(h).reshape(B * F, 32, self.npix)
        out = self.dec_final(self.dec1(h)).reshape(B, F, self.npix)
        if dataset_id is not None:
            out = self.film(out, dataset_id)
        return out


class FFM_OSLO_Autoencoder(nn.Module):
    """One SHARED-parameter inverse FiLM at the encoder input — zero extra parameters.

    With FiLM on the decoder output only, the colouration stays in x, the encoder takes it in, and it
    lands in z — but z is exactly what the downstream geometry->latent stage needs. Output-side FiLM
    only makes the reconstruction loss line up; it does not clean z.

    Here     x_norm = (x - beta)/gamma        <- remove colouration
             z      = encoder(x_norm)
             out    = gamma * decoder(z) + beta   <- back to the original space, loss still on raw

    The SAME FiLM parameters are used. Together with centring this also kills the trivial degenerate
    direction "scale gamma up, scale the network down". Set tie_encoder_film=False to disable.
    """
    def __init__(self, nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                 num_datasets=1, num_bins=93, tie_encoder_film=True):
        super().__init__()
        self.encoder = Isolated_OSLO_Encoder(nside, latent_dim_per_freq)
        self.decoder = Conditioned_OSLO_Decoder(nside, latent_dim_per_freq,
                                                freq_embed_dim, num_datasets, num_bins)
        self.tie_encoder_film = tie_encoder_film

    def forward(self, x, dataset_id=None):
        if dataset_id is not None and self.tie_encoder_film:
            x = self.decoder.film.inverse(x, dataset_id)
        z = self.encoder(x)
        return self.decoder(z, dataset_id), z


print('model defined (FFM + FiLM)')

In [ ]:
# ==========================================================================
# Cell 3 — 受试者、折、目标、耳廓点云（全部对齐到 FFM_3_2）
# ==========================================================================
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
L = np.load(PATHS['latents']); O = np.load(PATHS['oof'])
assert str(O['latents_subj_hash']) == str(L['subj_hash']) and str(O['latents_mesh_hash']) == str(L['mesh_hash']), \
    'oof_ridge.npz 与 latent 缓存不是同一批'
IDX = {s: i for i, s in enumerate(L['sid'])}
DEV = [str(s) for s in O['dev']]; NS = len(DEV)
LOCKED = {str(s) for s in L['locked_sids']}
assert set(DEV) <= {str(s) for s in L['eval_sids']} and not (set(DEV) & LOCKED)
FOLD = O['fold_of_subject']; FOLDS = sorted(set(FOLD.tolist()))
SUBJ_OF_ROW = np.repeat(np.arange(NS), 2); SIDE_OF_ROW = np.tile([0, 1], NS)
Y = np.stack([L['Z_ear'][IDX[s], side].reshape(-1) for s in DEV for side in (0, 1)]).astype(np.float64)
N_BINS = L['Z_ear'].shape[2]
RIDGE = {f"ridge: {str(O['best'])}": O['pred_best'].astype(np.float64), 'ridge: simple': O['pred_simple'].astype(np.float64)}

def cv_r2(pred):
    """与 FFM_3_2 的 run_cv 同口径：分母用每折训练集的均值"""
    sse = sst = 0.0
    for f in FOLDS:
        va = np.flatnonzero(FOLD[SUBJ_OF_ROW] == f); tr = np.flatnonzero(FOLD[SUBJ_OF_ROW] != f)
        my = Y[tr].mean(0); sse += ((Y[va] - pred[va]) ** 2).sum(); sst += ((Y[va] - my) ** 2).sum()
    return 1 - sse / sst

# 对齐检查：用导出的预测重算 ridge 的 CV-R²，必须与 FFM_3_2 的 results.json 一致
rj = os.path.join(os.path.dirname(PATHS['oof']), 'results.json')
if os.path.exists(rj):
    ref = json.load(open(rj))['cv_r2'][str(O['best'])]
    got = cv_r2(RIDGE[f"ridge: {str(O['best'])}"])
    assert abs(got - ref) < 1e-4, f'行或折没对齐：重算 {got:.6f} vs FFM_3_2 {ref:.6f}'
    print(f'对齐检查：ridge CV-R² 重算 {got:+.4f} = FFM_3_2 {ref:+.4f} ✓')
else:
    print('★ 没找到 FFM_3_2 的 results.json，跳过对齐检查')

with open(os.path.join(PATHS['samples'], 'manifest.csv'), newline='') as f:
    MAN = {f"SON_{r['subject']}" for r in csv.DictReader(f)}
assert hashlib.md5(','.join(sorted(MAN)).encode()).hexdigest()[:12] == str(L['mesh_hash'])
FLIP = np.array([1.0, -1.0, 1.0], np.float32)
PTS = []
for s in DEV:
    d = np.load(os.path.join(PATHS['samples'], f'{s[4:]}.npz'))
    PTS.append(d['earL_pts'] - d['ear_left_mm'])                       # 左耳，相对耳道
    PTS.append(d['earR_pts'] - d['ear_right_mm'] * FLIP)               # 右耳：点已镜像，耳道在这里镜像
PTS = torch.as_tensor(np.stack(PTS).astype(np.float32), device=device)  # (2NS, 20000, 3)
assert (PTS[:, :, 1].median(1).values > 0).all(), '镜像后耳廓应在 +y 一侧'
print(f'{NS} 人 / {2*NS} 只耳 | 折 {len(FOLDS)} | 点云 {tuple(PTS.shape)} | {device}')

In [ ]:
# ==========================================================================
# Cell 4 — PointNet
# ==========================================================================
import torch.nn as nn

class PointNetReg(nn.Module):
    def __init__(self, variant, k_out):
        super().__init__()
        self.variant = variant
        if variant == 'rff':
            g = torch.randn(CFG['RFF_M'], 3, generator=torch.Generator().manual_seed(12345))
            B = torch.cat([s * g for s in CFG['RFF_SIGMAS']])               # 固定，不训练
            self.register_buffer('B', B)
            d_in = 2 * len(B)
        else:
            d_in = 3
        layers, d = [], d_in
        for w in CFG['WIDTHS']:
            layers += [nn.Linear(d, w), nn.GELU()]; d = w
        self.point_mlp = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.Linear(2 * d, CFG['HEAD']), nn.GELU(), nn.Dropout(CFG['DROPOUT']),
                                  nn.Linear(CFG['HEAD'], k_out))

    def encode_points(self, p):                                              # p: (B, N, 3) mm
        if self.variant == 'rff':
            a = 2 * math.pi * p @ self.B.T
            return torch.cat([torch.cos(a), torch.sin(a)], -1)
        return p / CFG['XYZ_SCALE']

    def forward(self, p):
        h = self.point_mlp(self.encode_points(p))                            # (B, N, C)
        return self.head(torch.cat([h.max(1).values, h.mean(1)], -1))        # 对点的顺序和数量不敏感

for v in CFG['VARIANTS']:
    n = sum(p.numel() for p in PointNetReg(v, CFG['K_PC']).parameters())
    print(f'{v:4s}: {n:,} 个可训练参数（训练样本约 {int(2*NS*0.9)} 只耳）')

In [ ]:
# ==========================================================================
# Cell 5 — 一折的训练（epoch 固定；验证分只记录、不用于选择）
# ==========================================================================
def fold_pca(tr):
    my = Y[tr].mean(0); U, s, Vt = np.linalg.svd(Y[tr] - my, full_matrices=False)
    k = min(CFG['K_PC'], int((s > s[0] * 1e-9).sum())); V = Vt[:k].T
    return my, V, (Y[tr] - my) @ V

EVAL_IDX = [torch.arange(i * CFG['N_PTS'], (i + 1) * CFG['N_PTS'], device=device) for i in range(CFG['N_EVAL_DRAWS'])]
assert CFG['N_PTS'] * CFG['N_EVAL_DRAWS'] <= PTS.shape[1]

@torch.no_grad()
def predict(model, rows, my, V, scale):
    """模型输出的是 主成分坐标 / scale，这里乘回 scale 再还原成 z"""
    model.eval()
    out = np.zeros((len(rows), V.shape[1]))
    for idx in EVAL_IDX:                                    # 点已随机打乱，前 k 个即均匀子样本
        for b in range(0, len(rows), 64):
            r = torch.as_tensor(rows[b:b + 64], device=device)
            out[b:b + 64] += model(PTS[r][:, idx]).double().cpu().numpy()
    return my + (scale * out / len(EVAL_IDX)) @ V.T

def r2_rows(pred, rows, my):
    return 1 - ((Y[rows] - pred) ** 2).sum() / ((Y[rows] - my) ** 2).sum()

def train_fold(variant, f, shuffled):
    va = np.flatnonzero(FOLD[SUBJ_OF_ROW] == f); tr = np.flatnonzero(FOLD[SUBJ_OF_ROW] != f)
    my, V, T = fold_pca(tr)
    if shuffled:                                            # 训练集内按【人】打乱 z，两只耳一起换
        rng = np.random.default_rng(CFG['SEED'] + 1000 + f)
        subs = np.unique(SUBJ_OF_ROW[tr]); perm = dict(zip(subs, rng.permutation(subs)))
        pos = {r: i for i, r in enumerate(tr)}
        T = T[[pos[2 * perm[SUBJ_OF_ROW[r]] + SIDE_OF_ROW[r]] for r in tr]]
    scale = float(np.sqrt((T ** 2).sum(1).mean()))
    Tt = torch.as_tensor(T / scale, dtype=torch.float32, device=device)
    torch.manual_seed(CFG['SEED'] + 100 * f + (1 if shuffled else 0) + (7 if variant == 'rff' else 0))
    model = PointNetReg(variant, V.shape[1]).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['LR'], weight_decay=CFG['WD'])
    steps = CFG['EPOCHS'] * math.ceil(len(tr) / CFG['BATCH'])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG['LR'], total_steps=steps)
    curve = []
    trt = torch.as_tensor(tr, device=device)
    for ep in range(1, CFG['EPOCHS'] + 1):
        model.train()
        order = torch.randperm(len(tr), device=device)
        for b in range(0, len(tr), CFG['BATCH']):
            j = order[b:b + CFG['BATCH']]
            pick = torch.randint(0, PTS.shape[1], (len(j), CFG['N_PTS']), device=device)
            p = torch.gather(PTS[trt[j]], 1, pick[..., None].expand(-1, -1, 3))
            sh = torch.randn(len(j), 1, 3, device=device)
            p = p + CFG['JITTER_MM'] * sh / sh.norm(dim=-1, keepdim=True)
            loss = ((model(p) - Tt[j]) ** 2).sum(1).mean()
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        if ep % CFG['EVAL_EVERY'] == 0 or ep == CFG['EPOCHS']:
            ptr = predict(model, tr, my, V, scale); pva = predict(model, va, my, V, scale)
            # 训练 R² 对照的是模型实际拟合的目标（打乱时就是打乱后的 z）
            Ytr_fit = my + (T @ V.T)
            curve.append((ep, 1 - ((Ytr_fit - ptr) ** 2).sum() / ((Ytr_fit - my) ** 2).sum(), r2_rows(pva, va, my)))
    return pva, va, np.array(curve)

In [ ]:
# ==========================================================================
# Cell 6 — 跑全部折：每个变体 × (真实目标, 打乱对照)
# ==========================================================================
PN, CURVES = {}, {}
t0 = time.time()
for variant in CFG['VARIANTS']:
    for shuffled in ([False, True] if CFG['SHUFFLE_CONTROL'] else [False]):
        name = f"PointNet {variant}" + (" [打乱对照]" if shuffled else "")
        pred = np.zeros_like(Y); curves = []
        for f in FOLDS:
            pva, va, cv = train_fold(variant, f, shuffled)
            pred[va] = pva; curves.append(cv)
        PN[name] = pred; CURVES[name] = np.stack(curves)
        print(f'{name:28s} CV-R²(z) {cv_r2(pred):+.4f}   最后一个 epoch 的训练 R² {CURVES[name][:, -1, 1].mean():.3f}'
              f'   ({time.time()-t0:.0f} s)')
for n, p in RIDGE.items():
    print(f'{n:28s} CV-R²(z) {cv_r2(p):+.4f}')

In [ ]:
# ==========================================================================
# Cell 7 — 学习过程（只看，不用来挑 epoch）
# ==========================================================================
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
best_ridge = cv_r2(RIDGE[f"ridge: {str(O['best'])}"])
for name, c in CURVES.items():
    ep = c[0, :, 0]; ls = '--' if '打乱' in name else '-'
    ax[0].plot(ep, c[:, :, 1].mean(0), ls, label=name)
    ax[1].plot(ep, c[:, :, 2].mean(0), ls, label=name)
ax[1].axhline(best_ridge, c='k', lw=1, label='ridge 最好配置（FFM_3_2）'); ax[1].axhline(0, c='grey', lw=.6)
ax[0].set_title('训练集 R²（模型实际拟合的目标）'); ax[1].set_title('留出折 R²(z)，各折平均')
for a in ax: a.set_xlabel('epoch'); a.grid(alpha=.3); a.legend(fontsize=7)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'curves.png'), dpi=150); plt.show()
print('读法：训练 R² 高而留出 R² 不高 = 记忆；打乱对照的训练 R² 也很高 = 网络能背下任意标签。')
print(f"结论只取 epoch {CFG['EPOCHS']}（事先固定），不取曲线最高点。")

In [ ]:
# ==========================================================================
# Cell 8 — 端到端：pooled dB · LSD · 收回比例 · 认人；与 ridge 逐人配对比较
# ==========================================================================
import healpy as hp
ck = torch.load(PATHS['ckpt'], map_location=device, weights_only=False)
model_ae = FFM_OSLO_Autoencoder(nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                                num_datasets=4, num_bins=N_BINS, tie_encoder_film=True).to(device)
model_ae.load_state_dict(ck['model_state_dict']); model_ae.eval(); del ck

def find_dir(root, target):
    c = []
    for dp, _, fns in os.walk(root):
        if '__MACOSX' in dp: continue
        if os.path.basename(dp) == target: c.append((sum(f.endswith('.npy') for f in fns), dp))
    c.sort(reverse=True); return c[0][1] if c else None
if PATHS['sonicom'].endswith('.zip'):
    SON_OUT = '/content/dataset/sonicom'; os.makedirs(SON_OUT, exist_ok=True)
    if find_dir(SON_OUT, 'npy_left') is None:
        with zipfile.ZipFile(PATHS['sonicom']) as z: z.extractall(SON_OUT)
else:
    SON_OUT = PATHS['sonicom']
DIRS = (find_dir(SON_OUT, 'npy_left'), find_dir(SON_OUT, 'npy_right'))
_th, _ph = hp.pix2ang(16, np.arange(3072)); MIRROR = hp.ang2pix(16, _th, (-_ph) % (2 * np.pi))
MEAS = (90 - np.degrees(_th)) >= -45
def load_ear(sid, side):
    x = np.load(os.path.join(DIRS[side], f"{sid}_{('left', 'right')[side]}_full.npy")).astype(np.float32)[:N_BINS]
    return x[:, MIRROR] if side == 1 else x

AE_PART = dict(zip(L['sid'], L['ae_part']))
POOL = [str(s) for s in L['sid'] if AE_PART[s] in ('train', 'val')]
assert not set(POOL) & LOCKED
X = np.zeros((2 * NS, N_BINS, 3072), np.float32); SUM = np.zeros((N_BINS, 3072)); dev_pos = {s: k for k, s in enumerate(DEV)}
for s in POOL:
    for side in (0, 1):
        x = load_ear(s, side); SUM += x
        if s in dev_pos: X[2 * dev_pos[s] + side] = x
FREQS = np.arange(N_BINS) * 44100 / 256
NOTCH = (FREQS >= CFG['NOTCH_HZ'][0]) & (FREQS <= CFG['NOTCH_HZ'][1])

@torch.no_grad()
def decode_rows(Zrows):
    out = np.zeros((len(Zrows), N_BINS, 3072), np.float32)
    for b in range(0, len(Zrows), 16):
        z = torch.from_numpy(Zrows[b:b + 16].reshape(-1, N_BINS, 64).astype(np.float32)).to(device)
        out[b:b + 16] = model_ae.decoder(z, torch.full((len(z),), CFG['DID'], device=device)).cpu().numpy()
    return out

@torch.no_grad()
def retrieval(P):
    ranks = []
    for side in (0, 1):
        r = np.flatnonzero(SIDE_OF_ROW == side)
        a = torch.from_numpy(P[r].reshape(len(r), -1)).to(device); b = torch.from_numpy(X[r].reshape(len(r), -1)).to(device)
        d2 = (a * a).sum(1)[:, None] + (b * b).sum(1)[None] - 2 * a @ b.T
        ranks.append((d2 < d2.diagonal()[:, None]).sum(1).cpu().numpy() + 1)
    return np.concatenate(ranks)

def per_ear(P):
    e2 = (P - X) ** 2; d = np.sqrt(e2.mean(1))
    return dict(mse=e2.mean((1, 2)), mse_n=e2[:, NOTCH].mean((1, 2)), lsd=d.mean(1), lsd_m=d[:, MEAS].mean(1))

def subj(v): return v.reshape(NS, 2).mean(1)                     # 每人两只耳的平均

# ④ 基线（每折去掉留出的人）与 ① AE 重建
BASE = {k: np.zeros(2 * NS) for k in ('mse', 'mse_n', 'lsd', 'lsd_m')}
for f in FOLDS:
    va = np.flatnonzero(FOLD[SUBJ_OF_ROW] == f)
    mu = (SUM - X[va].sum(0)) / (2 * len(POOL) - len(va))
    e2 = (X[va] - mu) ** 2; d = np.sqrt(e2.mean(1))
    BASE['mse'][va] = e2.mean((1, 2)); BASE['mse_n'][va] = e2[:, NOTCH].mean((1, 2))
    BASE['lsd'][va] = d.mean(1); BASE['lsd_m'][va] = d[:, MEAS].mean(1)
_P = decode_rows(Y); REC = per_ear(_P); _rk = retrieval(_P); del _P

def summary(pe):
    return dict(db=float(np.sqrt(subj(pe['mse']).mean())), db_notch=float(np.sqrt(subj(pe['mse_n']).mean())),
                lsd=float(pe['lsd'].mean()), lsd_meas=float(pe['lsd_m'].mean()))
B, Rr = summary(BASE), summary(REC)
print(f'认人自检（AE 重建）：rank-1 {np.mean(_rk == 1):.1%}')
assert np.mean(_rk == 1) > 0.9, '认人自检失败：AE 重建都认不出本人'

ROWS, PE = {}, {}
for name, pred in list(RIDGE.items()) + list(PN.items()):
    P = decode_rows(pred); pe = per_ear(P); rk = retrieval(P); del P
    sm = summary(pe); PE[name] = pe
    ROWS[name] = dict(r2=float(cv_r2(pred)), **sm,
                      recovered=(B['db'] - sm['db']) / (B['db'] - Rr['db']),
                      recovered_notch=(B['db_notch'] - sm['db_notch']) / (B['db_notch'] - Rr['db_notch']),
                      rank1=float(np.mean(rk == 1)), median_rank=float(np.median(rk)))

hdr = f"{'':30s}{'CV-R²(z)':>9s}{'pooled':>8s}{'6–12k':>8s}{'LSD':>7s}{'LSD≥-45°':>10s}{'收回':>7s}{'认人':>7s}{'中位名次':>8s}"
print(hdr)
print(f"{'④ 均值 HRTF 基线':30s}{'':>9s}{B['db']:8.3f}{B['db_notch']:8.3f}{B['lsd']:7.3f}{B['lsd_meas']:10.3f}{'0%':>7s}")
for n, r in ROWS.items():
    print(f"{n:30s}{r['r2']:+9.4f}{r['db']:8.3f}{r['db_notch']:8.3f}{r['lsd']:7.3f}{r['lsd_meas']:10.3f}"
          f"{r['recovered']:7.1%}{r['rank1']:7.1%}{r['median_rank']:8.0f}")
print(f"{'① AE 重建（下限）':30s}{'':>9s}{Rr['db']:8.3f}{Rr['db_notch']:8.3f}{Rr['lsd']:7.3f}{Rr['lsd_meas']:10.3f}{'100%':>7s}")
print(f'认人随机水平 rank-1 {1/NS:.1%}，中位名次 {(NS+1)/2:.0f}')

# ---- 逐人配对：PointNet − ridge 最好配置，bootstrap 受试者 ----
ref_name = f"ridge: {str(O['best'])}"
rng = np.random.default_rng(CFG['SEED'])
BOOT = rng.integers(0, NS, (CFG['N_BOOT'], NS))
print(f'\n与 {ref_name} 逐人配对（负数 = 比 ridge 好），95% bootstrap 区间，{CFG["N_BOOT"]} 次重采样受试者：')
PAIRED = {}
for name in PN:
    a, b = subj(PE[name]['mse']), subj(PE[ref_name]['mse'])
    la, lb = subj(PE[name]['lsd']), subj(PE[ref_name]['lsd'])
    d_db = np.sqrt(a[BOOT].mean(1)) - np.sqrt(b[BOOT].mean(1)); d_lsd = la[BOOT].mean(1) - lb[BOOT].mean(1)
    PAIRED[name] = dict(d_db=float(np.sqrt(a.mean()) - np.sqrt(b.mean())), ci_db=list(np.percentile(d_db, [2.5, 97.5])),
                        d_lsd=float(la.mean() - lb.mean()), ci_lsd=list(np.percentile(d_lsd, [2.5, 97.5])))
    p = PAIRED[name]
    print(f"  {name:28s} pooled {p['d_db']:+.3f} dB [{p['ci_db'][0]:+.3f}, {p['ci_db'][1]:+.3f}]   "
          f"LSD {p['d_lsd']:+.3f} dB [{p['ci_lsd'][0]:+.3f}, {p['ci_lsd'][1]:+.3f}]")

In [ ]:
# ==========================================================================
# Cell 9 — 导出
# ==========================================================================
out = dict(cfg=CFG, exploratory=True, note='FFM_3_2 stopping rule already triggered; this run does not replace it',
           n_subjects=NS, folds=len(FOLDS), baseline=B, recon=Rr, rows=ROWS, paired_vs_ridge=PAIRED,
           curves={k: v.tolist() for k, v in CURVES.items()},
           latents_subj_hash=str(L['subj_hash']), latents_mesh_hash=str(L['mesh_hash']))
with open(os.path.join(OUT_DIR, 'results.json'), 'w') as f:
    json.dump(out, f, indent=1, ensure_ascii=False, default=float)
print(f'-> {OUT_DIR}/results.json, curves.png')

## 读结果之前

1. **Cell 3 对齐检查**必须通过：ridge 的 CV-R² 重算值要与 FFM_3_2 一致，否则行或折错位，后面全部作废。
2. **Cell 7**：打乱对照的训练 R² 如果和真实目标一样高，说明网络有能力背下任意标签——此时留出分数是唯一可信的读数。
3. **Cell 8 配对区间**：只有区间整体在 0 以下，才能说 PointNet 比 ridge 好；跨 0 就是「没有可分辨的差别」。
4. LSD 是文献口径（频率上 RMS、方向上算术平均），总是小于 pooled RMSE。和论文比较仍要核对频带、方向网格、数据库，
   并且同时报群体均值基线——这正是多数论文省略的那一列。
5. **探索性**：停止规则在 FFM_3_2 已经触发。这里无论结果如何，都写成「训练出的特征 vs 随机特征」的补充比较。